# MMML end-to-end in one notebook — a CGenFF residue

**Clean → split → train → evaluate → simulate → analyse**, driving MMML's CLI
functions and internal APIs directly from Python.

Every stage below has a `mmml <command>` CLI equivalent; the notebook calls the
*same functions the CLI calls*, so you get the CLI workflow with notebook
introspection (inspect intermediates, plot inline, tweak and re-run).

| Stage | CLI command | Python entrypoint used here |
|-------|-------------|------------------------------|
| Build residue | `mmml make-res --res ACO` | `mmml.cli.make.make_res.main_loop` |
| Clean + split | `mmml fix-and-split` | `mmml.cli.misc.fix_and_split.fix_and_split_data` |
| Train | `mmml physnet-train` | `mmml.cli.make.make_training.run_notebook` |
| Evaluate | `mmml physnet-evaluate` | ASE calc + `mmml.analysis.npz_comparison` |
| Simulate | `mmml md-system` / `mmml run` | `create_calculator_from_checkpoint` + ASE MD |
| Analyse | `mmml compare-npz` | `mmml.analysis.npz_comparison.plot_comparison` |

> Discover any command with `!mmml commands` or `!mmml <cmd> --help`, and see
> resolved checkpoints/CHARMM paths with `!mmml env`.


## 0. Environment
`x64` is required — MM/QM energies need double precision.

In [ ]:
import os, json, argparse
from pathlib import Path
import numpy as np

import jax
jax.config.update("jax_enable_x64", True)

import mmml
print("mmml:", getattr(mmml, "__version__", "dev"))
print("jax backend:", jax.default_backend(), jax.devices())

# Work in a scratch dir so the residue's pdb/psf/xyz/res/dcd folders land here.
WORK = Path("cgenff_e2e").resolve()
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print("cwd:", WORK)

## 1. Build the CGenFF residue

`make_res.main_loop` is exactly what `mmml make-res --res ACO` runs: it loads
PyCHARMM, reads the CGenFF RTF/PRM, generates the residue, builds coordinates,
and returns an `ase.Atoms`. It also creates the `pdb/ psf/ xyz/ res/ dcd/`
working directories.

Pick any residue from `mmml make-res --list-residues`. `ACO` = acetone.

In [ ]:
from mmml.interfaces.pycharmmInterface.import_pycharmm import ensure_pycharmm_loaded
ensure_pycharmm_loaded()
from mmml.cli.make import make_res

RES = "ACO"  # CGenFF RESI name; browse with:  !mmml make-res --list-residues --no-pager
args = argparse.Namespace(res=RES, list_residues=False, no_pager=True, skip_energy_show=True)
atoms = make_res.main_loop(args)

Z = atoms.get_atomic_numbers()
print(RES, "->", atoms.get_chemical_symbols(), "| R shape:", atoms.get_positions().shape)
print("wrote xyz/%s.xyz" % RES.lower())

## 2. Data cleaning + splitting

Training needs QM labels: an NPZ with `R` (coords), `Z` (atomic numbers),
`E` (energies), `F` (forces), optionally `D` (dipoles).

**How to produce real labels** (each is a `mmml` command you can run on the
geometries seeded from the residue above):
- `mmml normal-mode-sample` — perturb along vibrational modes to get geometries
- `mmml pyscf-evaluate` (or `mmml pyscf-dft`) — GPU DFT E/F/D labels
- `mmml sample-diverse-xyz` — SOAP-diverse subset → NPZ

`fix_and_split_data` (= `mmml fix-and-split`) does **both** jobs in one call:
fixes units (Hartree→eV, Hartree/Bohr→eV/Å, coords→Å, optional atom-ref
subtraction, sanity validation) **and** writes reproducible train/valid/test
splits.

### 2a. (Placeholder) make a tiny raw dataset so the notebook is self-contained

Replace this whole cell with your real QM NPZ. Here we jitter the residue
geometry and attach **dummy** energies/forces purely so the pipeline runs
end-to-end. Values are in Hartree / Hartree·Bohr⁻¹ to exercise unit-fixing.

In [ ]:
rng = np.random.default_rng(0)
N_RAW = 200
R0 = atoms.get_positions()                      # Angstrom
n_at = R0.shape[0]

R = R0[None] + rng.normal(0.0, 0.03, size=(N_RAW, n_at, 3))     # jittered coords (A)
E = (-193.0 + rng.normal(0, 1e-3, size=(N_RAW, 1))).astype(np.float64)   # fake Hartree
F = rng.normal(0.0, 1e-3, size=(N_RAW, n_at, 3)).astype(np.float64)      # fake Hartree/Bohr
Zb = np.repeat(Z[None, :], N_RAW, axis=0)

RAW = WORK / "raw_%s_efd.npz" % RES.lower()
np.savez(RAW, R=R.astype(np.float64), Z=Zb.astype(np.int64), E=E, F=F)
print("wrote", RAW, "| samples:", N_RAW, "| atoms:", n_at)

### 2b. Clean units + split

In [ ]:
from mmml.cli.misc.fix_and_split import fix_and_split_data

SPLITS = WORK / "training_data"
SPLITS.mkdir(exist_ok=True)

ok = fix_and_split_data(
    efd_file=RAW,
    output_dir=SPLITS,
    train_frac=0.8, valid_frac=0.1, test_frac=0.1, seed=42,
    coords_in="auto",       coords_out="angstrom",
    energy_in="hartree",    energy_out="ev",
    force_in="hartree_bohr", force_out="ev_angstrom",
    atomic_ref=None,        # e.g. "pbe0/def2-tzvp" to subtract per-atom reference energies
    skip_validation=False,  # runs sanity checks (min interatomic distance, unit heuristics, ...)
    verbose=True,
)
assert ok

train_npz = SPLITS / "energies_forces_dipoles_train.npz"
valid_npz = SPLITS / "energies_forces_dipoles_valid.npz"
test_npz  = SPLITS / "energies_forces_dipoles_test.npz"
for p in (train_npz, valid_npz, test_npz):
    n = len(np.load(p)["E"])
    print(f"{p.name:40s} {n:5d} samples")

## 3. Train a PhysNet (E/F) model

`run_notebook` is the notebook-friendly wrapper around `mmml physnet-train`.
Passing `valid_data` uses the full valid file (no internal resplit). Returns the
EMA params, a **portable JSON checkpoint** (`params_path`), and the Orbax run
dir. Keep epochs tiny here for a smoke run — scale up for real training.

In [ ]:
from mmml.cli.make import make_training

CKPTS = WORK / "ckpts"; CKPTS.mkdir(exist_ok=True)

ema_params, params_path, run_ckpt_dir = make_training.run_notebook(
    data=str(train_npz),
    valid_data=str(valid_npz),   # full valid file used; do NOT also set n_train/n_valid
    ckpt_dir=str(CKPTS),
    tag=RES.lower(),
    model=None,
    seed=42,
    batch_size=1,
    num_epochs=5,                # smoke run; use hundreds+ for real fits
    learning_rate=1e-3,
    energy_weight=1,
    objective="valid_loss",
    restart=None,
    num_atoms=None,
    # --- model hyperparameters ---
    features=64,
    max_degree=0,
    num_basis_functions=32,
    num_iterations=3,
    n_res=2,
    cutoff=8.0,
    max_atomic_number=28,
)
print("portable checkpoint:", params_path)
print("orbax run dir      :", run_ckpt_dir)

## 4. Evaluate on the held-out test set

We build the same ASE calculator used for simulation from the portable
checkpoint, predict E/F on the test frames, and score with
`mmml.analysis.npz_comparison` (the metric core behind `mmml compare-npz` /
`mmml physnet-evaluate`).

CLI equivalent:
`mmml physnet-evaluate --checkpoint <run_ckpt_dir> --data energies_forces_dipoles_test.npz`

In [ ]:
from mmml.interfaces.calculators.simple_inference import create_calculator_from_checkpoint
from ase import Atoms

calc = create_calculator_from_checkpoint(str(params_path), cutoff=8.0)

test = np.load(test_npz)
Rte, Zte, Ete, Fte = test["R"], test["Z"], test["E"], test["F"]
Zrow = Zte[0] if Zte.ndim == 2 else Zte

E_pred, F_pred = [], []
for i in range(len(Ete)):
    a = Atoms(numbers=Zrow, positions=Rte[i])
    a.calc = calc
    E_pred.append(a.get_potential_energy())
    F_pred.append(a.get_forces())
E_pred = np.array(E_pred).reshape(-1, 1)
F_pred = np.array(F_pred)
print("predicted", len(E_pred), "frames")

In [ ]:
from mmml.analysis.npz_comparison import compute_scalar_metrics, compute_force_metrics

# Both take (predictions, targets). compute_scalar_metrics -> ScalarMetrics (attrs);
# compute_force_metrics -> dict.
emet = compute_scalar_metrics(E_pred, Ete)
fmet = compute_force_metrics(F_pred, Fte)
print("Energy  MAE=%.4f  RMSE=%.4f  R2=%.4f" % (emet.mae, emet.rmse, emet.r2))
print("Forces  MAE=%.4f  RMSE=%.4f  R2=%.4f" % (fmet["mae"], fmet["rmse"], fmet["r2"]))
print("(dummy labels -> meaningless numbers; real QM data gives real metrics)")

## 5. Simulate the residue with the ML potential

The calculator plugs straight into ASE. Here: FIRE minimisation, then NVE
`VelocityVerlet` on the single CGenFF residue — a fast in-notebook check that the
trained potential drives stable dynamics.

For production MD (periodic liquid boxes, MM/ML hybrids, PyCHARMM/JAX-MD
backends) use the `mmml md-system` CLI — e.g.
`!mmml md-system --help` or one of the `md_system.*.example.yaml` presets in
`mmml/cli/run/`.

In [ ]:
from ase.optimize import FIRE
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.verlet import VelocityVerlet
from ase import units

sim = atoms.copy()
sim.calc = calc

FIRE(sim, logfile=None).run(fmax=0.5, steps=50)
print("minimised E = %.4f eV" % sim.get_potential_energy())

MaxwellBoltzmannDistribution(sim, temperature_K=300.0)
dyn = VelocityVerlet(sim, timestep=0.5 * units.fs)

traj = []
def _log():
    e = sim.get_potential_energy()
    ke = sim.get_kinetic_energy()
    traj.append((e, ke, e + ke))
dyn.attach(_log, interval=10)
dyn.run(200)   # 200 steps * 0.5 fs = 100 fs

traj = np.array(traj)
print("NVE steps logged:", len(traj), "| total-E drift: %.3e eV" % (traj[-1,2]-traj[0,2]))

## 6. Analyse

### 6a. Energy conservation of the NVE run

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

t = np.arange(len(traj)) * 10 * 0.5   # fs
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(t, traj[:,0], label="potential")
ax.plot(t, traj[:,1], label="kinetic")
ax.plot(t, traj[:,2], label="total", lw=2, color="k")
ax.set_xlabel("time (fs)"); ax.set_ylabel("energy (eV)"); ax.legend(loc="best")
ax.set_title("%s NVE energy conservation" % RES)
fig.tight_layout(); fig.savefig(WORK/"nve_energy.png", dpi=120)
print("saved", WORK/"nve_energy.png"); plt.show()

### 6b. Model-vs-reference parity plots

Two ways: (1) `align_npz_arrays` + `plot_comparison` — the exact functions behind
`mmml compare-npz`; (2) the `mmml compare-npz` CLI on saved NPZs. We write a
prediction NPZ, then render inline.

In [ ]:
from mmml.analysis.npz_comparison import align_npz_arrays, compute_force_metrics

pred_npz = WORK / "test_pred.npz"
np.savez(pred_npz, R=Rte, Z=Zte, E=E_pred, F=F_pred)

# Direct parity plot from arrays we already hold (guaranteed to run):
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].scatter(Ete.reshape(-1), E_pred.reshape(-1), s=10, alpha=0.6)
lo, hi = Ete.min(), Ete.max()
ax[0].plot([lo, hi], [lo, hi], "k--", lw=1)
ax[0].set_xlabel("E ref (eV)"); ax[0].set_ylabel("E pred (eV)"); ax[0].set_title("Energy")
ax[1].scatter(Fte.reshape(-1), F_pred.reshape(-1), s=4, alpha=0.3)
fl = np.abs(Fte).max()
ax[1].plot([-fl, fl], [-fl, fl], "k--", lw=1)
ax[1].set_xlabel("F ref (eV/A)"); ax[1].set_ylabel("F pred (eV/A)"); ax[1].set_title("Forces")
fig.tight_layout(); fig.savefig(WORK/"parity.png", dpi=120); plt.show()

# Batch equivalent via the CLI (handles alignment + full report for you):
print(f"CLI equivalent:  !mmml compare-npz --reference {test_npz} --model {pred_npz}")

### 6c. Training curves from the checkpoint

`mmml extract-checkpoint-metrics` plots loss/MAE vs epoch straight from the
Orbax run directory.

In [ ]:
print("Training-curve plot from the run we just produced:")
print(f"  !mmml extract-checkpoint-metrics --checkpoint {run_ckpt_dir}")
# e.g.:  !mmml extract-checkpoint-metrics --checkpoint {run_ckpt_dir} --out {WORK/'metrics.png'}

---
## Recap — the whole pipeline as CLI one-liners

```bash
mmml make-res --res ACO                         # 1. build residue (pdb/psf/xyz)
mmml normal-mode-sample ...                     # 2a. geometries  (real data)
mmml pyscf-evaluate ...                         # 2a. QM labels   (real data)
mmml fix-and-split --efd raw.npz --output-dir training_data   # 2b. clean + split
mmml physnet-train --data training_data/..._train.npz \
     --valid-data training_data/..._valid.npz --ckpt-dir ckpts --tag aco
mmml physnet-evaluate --checkpoint ckpts/<run> --data ..._test.npz   # 4. evaluate
mmml md-system --help                           # 5. simulate (boxes / hybrid MD)
mmml compare-npz --reference ..._test.npz --model test_pred.npz     # 6. analyse
```

This notebook called the same functions those commands dispatch to, so you can
mix CLI convenience with full Python introspection. Swap the placeholder data in
§2a for real QM labels and scale up `num_epochs` for a production model.
